In [ ]:
import os
import glob

data_root = '/home/featurize/data/Task1_Liu/'

ct_root = data_root + 'CT'
mri_root = data_root + 'MR'

# 获取ct_root与mri_root下的所有文件
ct_files = glob.glob(os.path.join(ct_root, '*.npy'))
mri_files = glob.glob(os.path.join(mri_root, '*.npy'))
print(ct_files)

# 将前缀相同的视为一组，放入字典/CT/1BA001.npy与/MR/1BA001.npy
# "1BA001": {"CT": <ct_path>, "MR": <ct_path>}
data_dict = {}

for ct_file in ct_files:
    # 获取文件名
    ct_name = os.path.basename(ct_file)
    # 获取前缀
    prefix = ct_name.split('.')[0]
    # 将CT文件路径存入字典
    data_dict[prefix] = {"CT": ct_file}
for mri_file in mri_files:
    # 获取文件名
    mri_name = os.path.basename(mri_file)
    # 获取前缀
    prefix = mri_name.split('.')[0]
    # 将MR文件路径存入字典
    if prefix in data_dict:
        data_dict[prefix]["MR"] = mri_file
    else:
        data_dict[prefix] = {"MR": mri_file}
# 打印字典
for key, value in data_dict.items():
    print(f"{key}: {value}")
    


# 读取第一个样本，分别输出CT与MR的shape
import numpy as np

sample = list(data_dict.values())[0]
ct_path = sample["CT"]
mri_path = sample["MR"]

ct_data = np.load(ct_path)
mri_data = np.load(mri_path)

print(f"CT shape: {ct_data.shape}") # (64, 256, 256)
print(f"MR shape: {mri_data.shape}") # (64, 256, 256)


['/home/featurize/data/Task1_Liu/CT/1BC080.npy', '/home/featurize/data/Task1_Liu/CT/1BB096.npy', '/home/featurize/data/Task1_Liu/CT/1BC087.npy', '/home/featurize/data/Task1_Liu/CT/1BC066.npy', '/home/featurize/data/Task1_Liu/CT/1BC019.npy', '/home/featurize/data/Task1_Liu/CT/1BC014.npy', '/home/featurize/data/Task1_Liu/CT/1BC031.npy', '/home/featurize/data/Task1_Liu/CT/1BA001.npy', '/home/featurize/data/Task1_Liu/CT/1BC050.npy', '/home/featurize/data/Task1_Liu/CT/1BC062.npy', '/home/featurize/data/Task1_Liu/CT/1BA014.npy', '/home/featurize/data/Task1_Liu/CT/1BB062.npy', '/home/featurize/data/Task1_Liu/CT/1BA075.npy', '/home/featurize/data/Task1_Liu/CT/1BB071.npy', '/home/featurize/data/Task1_Liu/CT/1BA239.npy', '/home/featurize/data/Task1_Liu/CT/1BB189.npy', '/home/featurize/data/Task1_Liu/CT/1BA185.npy', '/home/featurize/data/Task1_Liu/CT/1BC027.npy', '/home/featurize/data/Task1_Liu/CT/1BC051.npy', '/home/featurize/data/Task1_Liu/CT/1BC049.npy', '/home/featurize/data/Task1_Liu/CT/1BB0

In [ ]:
# dataset类
import numpy as np
import torch
from torch.utils.data import Dataset

class CustomDataset(Dataset):
    def __init__(self, data_root, transform=None):
        self.data_root = data_root
        self.transform = transform
        
        self._load_data_path()
        
    def _load_data_path(self):
        self.data_dict = {}
        # 获取ct_root与mri_root下的所有文件
        ct_files = glob.glob(os.path.join(self.data_root, 'CT', '*.npy'))
        mri_files = glob.glob(os.path.join(self.data_root, 'MR', '*.npy'))

        for ct_file in ct_files:
            # 获取文件名
            ct_name = os.path.basename(ct_file)
            # 获取前缀
            prefix = ct_name.split('.')[0]
            # 将CT文件路径存入字典
            self.data_dict[prefix] = {"CT": ct_file}
            
            # 将depth存入字典          # ===== 以CT为准 ===== #
            ct_data = np.load(ct_file)
            self.data_dict[prefix]["depth"] = ct_data.shape[0]
        for mri_file in mri_files:
            # 获取文件名
            mri_name = os.path.basename(mri_file)
            # 获取前缀
            prefix = mri_name.split('.')[0]
            # 将MR文件路径存入字典
            if prefix in self.data_dict:
                self.data_dict[prefix]["MR"] = mri_file
            else:
                self.data_dict[prefix] = {"MR": mri_file}
        
        # 对于每个样本，检查CT和MR是否都存在，移除不完整的样本
        self.data_dict = {k: v for k, v in self.data_dict.items() if "CT" in v and "MR" in v}
        self.keys = list(self.data_dict.keys())
        
        # 求length # depth之和
        self.length = sum([self.data_dict[key]["depth"] for key in self.keys])
        
        # 维护一个表示取指定索引数据要到某个样本的某切片的dict
        # 最终每item取出的是一个2维切片
        self.index_dict = {}
        idx = 0
        for key in self.keys:
            depth = self.data_dict[key]["depth"]
            for i in range(depth):
                self.index_dict[idx] = {
                    "CT": self.data_dict[key]["CT"], 
                    "MR": self.data_dict[key]["MR"], 
                    "slice_idx": i
                    }
                
                idx += 1
            
    def __len__(self):
        return self.length
    
    def __getitem__(self, idx):
        # 获取CT和MR的路径
        ct_path = self.index_dict[idx]["CT"]
        mri_path = self.index_dict[idx]["MR"]
        slice_idx = self.index_dict[idx]["slice_idx"]
        
        # 读取CT和MR数据
        ct_data = np.load(ct_path)
        mri_data = np.load(mri_path)
        
        # 获取指定切片
        ct_slice = ct_data[slice_idx, :, :]
        mri_slice = mri_data[slice_idx, :, :]
        
        # 转换为tensor
        ct_slice = torch.tensor(ct_slice, dtype=torch.float32)
        mri_slice = torch.tensor(mri_slice, dtype=torch.float32)
        
        if self.transform:
            ct_slice = self.transform(ct_slice)
            mri_slice = self.transform(mri_slice)
        
        return ct_slice, mri_slice
    
# 测试dataset
data_root = '/home/featurize/data/Task1_Liu/'
dataset = CustomDataset(data_root)
ct_mri = dataset[0]

print(f"CT shape: {ct_mri[0].shape}") # (256, 256)
print(f"MR shape: {ct_mri[1].shape}") # (256, 256)
    

                
    
        
    

CT shape: torch.Size([256, 256])
MR shape: torch.Size([256, 256])


In [1]:
import torch

# 查看PyTorch版本
print(torch.__version__)

# 查看CUDA是否可用
print(torch.cuda.is_available())

# 查看可用的CUDA设备数量
print(torch.cuda.device_count())

# 查看PyTorch支持的CUDA版本
print(torch.version.cuda)

2.2.2+cu121
True
1
12.1
